# 03 EModel Validation
Build an `EModelValidationScanConfig`, expand it with `GridScanGenerationTask`, register the campaign and single-coordinate `TaskConfig` entities in entitycore, and run the registered single config locally through OBI-One's task runner.

The task resolves the optimisation that produced the EModel (notebook 02) through provenance, rebuilds its BluePyEModel working directory, scores the optimised model against the targets with BluePyEModel and registers a `TaskResult` and `ValidationResult` entities on the EModel.

**Prerequisites:** `bluepyemodel`, NEURON 8.2.x with `nrnivmodl`, access to entitycore staging, and an EModel with resolvable optimisation provenance — either registered by the optimisation workflow (Task 2), or carrying a campaign scan config and an `emodel_optimization_output` asset (e.g. backfilled provenance).

## Imports

In [ ]:
# Optional imports used only by the commented Launch System reference cell below.
# from http import HTTPStatus
# import httpx
import json
import shutil
from pathlib import Path

from entitysdk import Client, ProjectContext, models
from obi_auth import get_token

import obi_one as obi
from obi_one.core.info import Info
from obi_one.core.run_tasks import run_task_for_single_config
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.from_id.emodel_from_id import EModelFromID
from obi_one.scientific.from_id.task_result_from_id import TaskResultFromID
from obi_one.scientific.tasks.emodel_building.task3_emodel_validation.blocks import (
    EModelInput,
    ValidationSettings,
    ValidationTargetEFeaturesInput,
)
from obi_one.scientific.tasks.emodel_building.task3_emodel_validation.config import (
    EModelValidationScanConfig,
)

## Connect to entitycore

In [ ]:
environment = "staging"
virtual_lab_id = obi.LAB_ID_STAGING_TEST  # staging test virtual lab
project_id = obi.PROJECT_ID_STAGING_TEST  # staging test project

token = get_token(environment=environment)
db_client = Client(
    api_url="https://staging.cell-a.openbraininstitute.org/api/entitycore",
    project_context=ProjectContext(virtual_lab_id=virtual_lab_id, project_id=project_id),
    token_manager=token,
)

## Pipeline handoff
Notebooks 01, 02 and 03 can be run in sequence: each writes the ID of the entity it registers to a shared handoff file, and the next notebook reads it.

In [ ]:
# Notebooks 01 -> 02 -> 03 hand their output entity IDs to the next stage through this file.
HANDOFF_PATH = Path("../../../../../../obi-output/emodel_building_handoff.json").resolve()


def read_handoff() -> dict:
    return json.loads(HANDOFF_PATH.read_text()) if HANDOFF_PATH.exists() else {}


def write_handoff(**ids: str | list[str]) -> None:
    HANDOFF_PATH.parent.mkdir(parents=True, exist_ok=True)
    HANDOFF_PATH.write_text(json.dumps(read_handoff() | ids, indent=2))
    print(f"Handoff {HANDOFF_PATH}: {read_handoff()}")


def generated_ids(activity_id, entity_type: str) -> list[str]:
    """IDs of the entities of ``entity_type`` generated by an execution activity."""
    activity = db_client.get_entity(entity_id=activity_id, entity_type=models.TaskActivity)
    return [str(e.id) for e in activity.generated or [] if e.type == entity_type]


## Build the scan config
- `emodel`: the EModel registered by notebook 02.
- `target_efeatures`: the e-feature extraction TaskResult to score against. With the e-model's optimisation targets (notebook 01's extraction), the held-out protocols are scored; with another (e.g. fresh) extraction, all its protocols are validation protocols by default.
- The morphology is always the exemplar the e-model was optimised on. To test another morphology, create an ME-model with it and validate that ME-model.
- `settings`: empty values fall back to the optimisation configuration (`validation_function`, `validation_threshold`).

In [ ]:
# Replace with real entity IDs from your staging project.
# The EModel registered by notebook 02 when it was run before this notebook.
EMODEL_ID = read_handoff().get("emodel_id", "<emodel-id-from-notebook-02>")
print(f"EModel: {EMODEL_ID}")
# The validation protocols are chosen automatically from the target e-features.
print(f"Validation protocols held out in 01/02: {read_handoff().get('validation_protocols')}")
# The optimisation targets (notebook 01's extraction); replace with another e-feature
# extraction TaskResult to validate against fresh data.
TARGET_EFEATURES_TASK_RESULT_ID = read_handoff().get(
    "extraction_task_result_id", "<extraction-task-result-id>"
)

scan_config = EModelValidationScanConfig(
    info=Info(
        campaign_name="EModel validation",
        campaign_description=f"BluePyEModel score validation of EModel {EMODEL_ID}.",
    ),
    emodel=EModelInput(emodel=EModelFromID(id_str=EMODEL_ID)),
    target_efeatures=ValidationTargetEFeaturesInput(
        task_result=TaskResultFromID(id_str=TARGET_EFEATURES_TASK_RESULT_ID)
    ),
    settings=ValidationSettings(
        validation_protocols_only=False,
        validation_function="from_optimisation",  # or "max_score" / "mean_score"
        validation_threshold=None,  # e.g. 5.0
    ),
)
scan_config

## Register the campaign and single TaskConfigs
The next cell removes only the previous local `obi-output/03_emodel_validation/grid_scan` directory; it does not delete entitycore entities. It does not run the validation.

In [ ]:
grid_scan = GridScanGenerationTask(
    form=scan_config,
    output_root="../../../../../../obi-output/03_emodel_validation/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
output_root = grid_scan.output_root_absolute
if output_root.parts[-3:] != ("obi-output", "03_emodel_validation", "grid_scan"):
    raise RuntimeError(f"Refusing to clean unexpected Task 3 output root: {output_root}")
if output_root.exists():
    shutil.rmtree(output_root)
    print(f"Deleted previous local Task 3 execution: {output_root}")

grid_scan.execute(db_client=db_client)
print(f"Registered campaign TaskConfig: {grid_scan.form.campaign.id}")
for config in grid_scan.single_configs:
    print(f"  Coordinate {config.idx}: {config.single_entity.id}")

## Run the validation locally
Stages the inputs, compiles mechanisms, recomputes the responses to all optimisation and validation protocols, scores them, plots scores and traces under `figures/`, writes `validation_results.json` and registers the outputs.

In [ ]:
from entitysdk.types import ActivityStatus, TaskActivityType

from obi_one.db_sdk import db_sdk


def create_execution_activity(single_config, activity_type: TaskActivityType):
    """Create the execution TaskActivity the launch service would create for this config.

    Registration links the generated entities to it, which is the provenance later
    stages (e.g. Task 3 validation) resolve.
    """
    return db_sdk.create_generic_activity(
        client=db_client,
        used=[single_config.single_entity],
        activity_type=activity_type,
        activity_status=ActivityStatus.running,
    )


def finalize_execution_activity(activity, status: ActivityStatus) -> None:
    db_sdk.finalize_activity(
        client=db_client,
        activity_id=activity.id,
        activity_type=models.TaskActivity,
        status=status,
    )


for single_config in grid_scan.single_configs:
    coordinate_root = Path(single_config.coordinate_output_root).resolve()
    activity = create_execution_activity(single_config, TaskActivityType.optimized_emodel_analysis_validation__execution)
    try:
        run_task_for_single_config(
            single_config,
            db_client=db_client,
            entity_cache=False,
            execution_activity_id=str(activity.id),
        )
    except Exception:
        finalize_execution_activity(activity, ActivityStatus.error)
        raise
    finalize_execution_activity(activity, ActivityStatus.done)
    summary = json.loads((coordinate_root / "validation_results.json").read_text())
    verdict = "PASSED" if summary["passed"] else "FAILED"
    print(f"Coordinate {single_config.idx}: {verdict} (score={summary['score']:.2f})")
    print(f"  settings: {summary['settings']}")
    for protocol, entry in summary["protocols"].items():
        role = "validation" if entry["validation"] else "optimisation"
        print(f"  {protocol:<20} {role:<12} score={entry['score']:.2f} passed={entry['passed']}")

## Inspect the registered ValidationResults

In [ ]:
results = db_client.search_entity(
    entity_type=models.ValidationResult,
    query={"validated_entity_id": EMODEL_ID},
).all()
for result in results:
    print(f"{result.name:<60} passed={result.passed}  id={result.id}")

## Optional Launch System execution (commented)

In [ ]:
# payload = {
#     "task_type": "emodel_validation",
#     "config_id": str(grid_scan.single_configs[0].single_entity.id),
# }
# response = api_client.post(url="/declared/task/launch", json=payload, timeout=300.0)
# if response.status_code != HTTPStatus.OK:
#     response.raise_for_status()
# print(response.json())